# 1 – Indexing: Dokumente in durchsuchbare Indizes überführen

Dieses Notebook führt die **Indexing-Pipeline** durch:

1. **Laden** – PDFs und Word-Dokumente werden eingelesen
2. **Chunking** – Der Text wird in überlappende Abschnitte zerlegt
3. **Vektor-Index** – Jeder Chunk wird zu einem Vektor → ChromaDB *(semantische Suche)*
4. **BM25-Index** – Dieselben Chunks für die lexikalische Suche *(optional)*

Schritt 1 und 2 sind der gemeinsame Stamm, Schritt 3 und 4 zwei unabhängige Senken daran:

```
Laden → Chunking ─┬─→ ChromaDB   (braucht das Embedding-Modell, dauert Minuten)
                  └─→ BM25       (braucht kein Modell, dauert Sekunden)
```

> **Hinweis:** Dieses Notebook muss nur einmal ausgeführt werden – oder wenn neue
> Dokumente dazukommen. Notebook 2 liest anschließend den fertigen Index.

## Benötigte Pakete

```bash
pip install -U langchain-core langchain-community langchain-text-splitters \
               langchain-openai langchain-chroma pymupdf docx2txt
# nur bei EMBEDDING_BACKEND = "huggingface":
pip install -U "langchain-huggingface[full]"
```

Das Extra `[full]` ist nötig, weil die Basis-Installation `sentence-transformers` nicht mehr mitbringt.

## Konfiguration

Die **einzige Zelle, die du anpassen musst**. Sie ist in allen Notebooks der Reihe
wortgleich – Abschnitte, die dieses Notebook nicht braucht, stören hier nicht.

In [1]:
# ============================================================
#  KONFIGURATION – in allen Notebooks der Reihe identisch
# ============================================================

# --- 1) Embedding-Umschalter --------------------------------
#     EIN Wert wählt Modell + Backend + Ansprache gemeinsam,
#     damit Name und Konvention nicht auseinanderdriften.
EMBEDDING = "bge-m3"                  # "bge-m3" | "e5-instruct" | "e5"

E5_INSTRUCT_TASK = "Given a web search query, retrieve relevant passages that answer the query"

EMBEDDING_PROFILES = {
    "bge-m3": {          # LM Studio, keine Präfixe
        "backend": "lmstudio",
        "model":   "text-embedding-bge-m3",
        "doc_prefix": "", "query_prefix": "",
    },
    "e5-instruct": {     # e5-large-instruct: Aufgaben-Satz an die Frage, Dokument roh
        "backend": "lmstudio",   # alternativ "huggingface" (model = "intfloat/multilingual-e5-large-instruct")
        "model":   "text-embedding-multilingual-e5-large-instruct",
        "doc_prefix": "", "query_prefix": f"Instruct: {E5_INSTRUCT_TASK}\nQuery: ",
    },
    "e5": {              # klassisches E5 (NICHT instruct): passage/query
        "backend": "lmstudio",
        "model":   "text-embedding-multilingual-e5-large",
        "doc_prefix": "passage: ", "query_prefix": "query: ",
    },
}

# Abgeleitet – der Rest des Notebooks nutzt nur diese drei:
_ep               = EMBEDDING_PROFILES[EMBEDDING]
EMBEDDING_BACKEND = _ep["backend"]
EMBEDDING_MODEL   = _ep["model"]
EMBEDDING_PROMPTS = {"doc": _ep["doc_prefix"], "query": _ep["query_prefix"]}

# --- 2) LLM-Backend (ab Notebook 2) --------------------------
LLM_BACKEND = "lmstudio"              # "lmstudio" | "deepinfra" | "openrouter"

# --- 3) API-Keys der Cloud-Anbieter --------------------------
#     Nur ausfüllen, wenn das jeweilige Backend genutzt wird.
#     Bitte den Key für dich behalten: nicht weitergeben, nicht committen,
#     nicht in geteilten Notebooks stehen lassen.
DEEPINFRA_API_KEY  = ""
OPENROUTER_API_KEY = ""

# --- 4) Pfade ------------------------------------------------
DOC_SOURCE_DIR = "./documents"        # Quell-Dokumente (PDF + DOCX)
DB_DIR         = "./chroma_db"        # Vektordatenbank
#BM25_DIR       = "./bm25_index"       # Lexikalischer Index
MODEL_PATH     = "./models"           # Modell-Cache (nur HuggingFace)
MANIFEST_PATH  = "./index_manifest.json"

COLLECTION_NAME = "langchain"         # muss in Notebook 1 und 2 gleich sein

# --- 5) Chunking (Notebook 1) --------------------------------
#     Ändert man das hier, muss der komplette Index neu gebaut werden.
CHUNK_SIZE    = 512
CHUNK_OVERLAP = 128                   # ~25 % Überlappung – gut für lange deutsche Sätze

# --- 6) Welche Indizes bauen? (Notebook 1) -------------------
BUILD_VECTOR_INDEX = True             # semantisch, braucht das Embedding-Modell
BUILD_BM25_INDEX   = False             # lexikalisch, braucht kein Modell

# --- 7) Retrieval (ab Notebook 2) ----------------------------
TOP_K = 10                            # Anzahl der Chunks pro Frage

# ============================================================
#  Backend-Details – normalerweise unverändert lassen
# ============================================================

LM_STUDIO_URL = "http://localhost:1234/v1"

LLM_CONFIG = {
    "lmstudio": {
        "base_url": LM_STUDIO_URL,
        "model":    "qwen/qwen3.5-9b",
        "api_key":  "lm-studio",                  # LM Studio prüft den Key nicht
    },
    "deepinfra": {
        "base_url": "https://api.deepinfra.com/v1/openai",
        "model":    "meta-llama/Llama-3.3-70B-Instruct-Turbo",
        "api_key":  DEEPINFRA_API_KEY,
    },
    "openrouter": {
        "base_url": "https://openrouter.ai/api/v1",
        "model":    "meta-llama/llama-3.3-70b-instruct",
        "api_key":  OPENROUTER_API_KEY,
    },
}

print(f"Embeddings: {EMBEDDING_BACKEND}  |  Vektor-Index: {BUILD_VECTOR_INDEX}  |  BM25-Index: {BUILD_BM25_INDEX}")

Embeddings: lmstudio  |  Vektor-Index: True  |  BM25-Index: False


## Imports

In [2]:
import os
import glob
import json
import shutil
from datetime import datetime

from langchain_community.document_loaders import PyMuPDFLoader, Docx2txtLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma

/var/folders/r6/w7htzvyn5czgkqd55n18p3y80000gn/T/ipykernel_12939/2956393172.py:7: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyMuPDFLoader, Docx2txtLoader
/Users/nataliajoerg/Documents/05_VSCode/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Schritt 1 – Dokumente laden (PDF & DOCX)

Zwei Details, die auf verschiedenen Betriebssystemen sonst still danebengehen:

- **Dateiendungen werden case-insensitiv geprüft.** `*.pdf` findet unter macOS und Linux
  keine Datei namens `Bericht.PDF` – unter Windows schon. Der Index wäre dann je nach
  Rechner unterschiedlich vollständig, ohne Fehlermeldung.
- **Die Dateiliste wird sortiert.** `glob` liefert je nach Dateisystem eine andere
  Reihenfolge; sortiert man, ist der Index reproduzierbar.

In [3]:
# Mapping: Dateiendung → passender LangChain-Loader
LOADERS = {
    ".pdf":  PyMuPDFLoader,
    ".docx": Docx2txtLoader,
}


def load_documents(source_dir: str):
    """Lädt alle unterstützten Dokumente (PDF, DOCX) aus einem Verzeichnis."""
    all_docs = []
    file_count = 0

    # Alle Dateien einsammeln, Endung kleingeschrieben vergleichen, dann sortieren
    candidates = sorted(
        path for path in glob.glob(os.path.join(source_dir, "*"))
        if os.path.splitext(path)[1].lower() in LOADERS
    )

    for filepath in candidates:
        ext = os.path.splitext(filepath)[1].lower()
        loader = LOADERS[ext](filepath)
        all_docs.extend(loader.load())
        print(f"  📄 {os.path.basename(filepath)}")
        file_count += 1

    print(f"\n✅ {len(all_docs)} Seiten/Abschnitte aus {file_count} Dokumenten geladen.")
    return all_docs, file_count

## Schritt 2 – Text in Chunks aufteilen

> ⚠️ **Hier entsteht die gemeinsame Grundlage beider Indizes.**
> Vektorsuche und BM25 dürfen nicht nur denselben Corpus sehen, sondern müssen auf
> **denselben Chunks** aufsetzen. Sonst liefern sie Treffer unterschiedlichen
> Zuschnitts, die sich in der Hybrid-Suche nicht mehr zusammenführen lassen.

In [4]:
def split_documents(docs, chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP):
    """Zerlegt Dokumente in überlappende Text-Chunks."""
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
    )
    chunks = splitter.split_documents(docs)
    print(f"✅ {len(chunks)} Chunks erzeugt (Größe: {chunk_size}, Overlap: {chunk_overlap})")
    return chunks

## Schritt 3 – Vektor-Index (ChromaDB)

Das Modell `multilingual-e5-large-instruct` erwartet **spezifische Präfixe**:

- Dokumente (beim Indexieren): `passage: ...`
- Suchanfragen (beim Retrieval): `query: ...`

Ohne diese Präfixe arbeitet das Modell deutlich unter seinem Niveau. LangChain setzt sie
nicht automatisch, deshalb der dünne Wrapper. Beide Backends laden dasselbe Modell –
LM Studio lediglich als **Q8_0-Quantisierung**.

In [5]:
class _Prefix:
    """Stellt die profil-typischen Präfixe voran (doc/query aus EMBEDDING_PROMPTS)."""
    _p = EMBEDDING_PROMPTS

    def embed_documents(self, texts: list[str]) -> list[list[float]]:
        return super().embed_documents([self._p["doc"] + t for t in texts])

    def embed_query(self, text: str) -> list[float]:
        return super().embed_query(self._p["query"] + text)


class PrefixOpenAIEmbeddings(_Prefix, OpenAIEmbeddings):
    """Für LM Studio (OpenAI-kompatibel)."""


def build_embeddings():
    """Erzeugt das Embedding-Modell passend zum aktiven Profil."""
    if EMBEDDING_BACKEND == "lmstudio":
        return PrefixOpenAIEmbeddings(
            model=EMBEDDING_MODEL,
            api_key="lm-studio",
            base_url=LM_STUDIO_URL,
            check_embedding_ctx_length=False,
        )

    if EMBEDDING_BACKEND == "huggingface":
        # Import erst hier, damit LM-Studio-Nutzer das Paket nicht brauchen
        from langchain_huggingface import HuggingFaceEmbeddings

        class PrefixHFEmbeddings(_Prefix, HuggingFaceEmbeddings):
            """Für lokal geladene sentence-transformers-Modelle."""

        return PrefixHFEmbeddings(model_name=EMBEDDING_MODEL, cache_folder=MODEL_PATH)

    raise ValueError(f"Unbekanntes EMBEDDING_BACKEND: {EMBEDDING_BACKEND}")

In [6]:
def create_vectorstore(chunks, db_dir=DB_DIR):
    """Erzeugt Embeddings und speichert sie in einer lokalen ChromaDB."""
    # Alten Index löschen, damit keine veralteten Vektoren übrig bleiben
    if os.path.exists(db_dir):
        shutil.rmtree(db_dir)
        print("🗑️  Alter Vektor-Index gelöscht.")

    # Das Embedding-Modell wird erst hier geladen – wer nur BM25 baut, braucht es nicht
    embeddings = build_embeddings()
    print(f"🧠 Erstelle Embeddings ({EMBEDDING_BACKEND}) und speichere in ChromaDB...")

    vectorstore = Chroma.from_documents(
        documents=chunks,
        embedding=embeddings,
        collection_name=COLLECTION_NAME,
        persist_directory=db_dir,
    )
    print(f"✨ Vektor-Index gespeichert in '{db_dir}'")
    return vectorstore

## Schritt 5 – Manifest schreiben

Eine kleine JSON-Datei hält fest, **womit** dieser Index gebaut wurde. Notebook 2 gibt sie
beim Start nur aus – es prüft nichts und bricht nichts ab. Der Abgleich mit der eigenen
Konfiguration bleibt deine Aufgabe, aber du siehst schwarz auf weiß, was in der Datenbank
liegt.

In [7]:
def write_manifest(n_docs, n_chunks, path=MANIFEST_PATH):
    """Dokumentiert die Parameter, mit denen die Indizes gebaut wurden."""
    manifest = {
        "erstellt":          datetime.now().isoformat(timespec="seconds"),
        "quellordner":       DOC_SOURCE_DIR,
        "dokumente":         n_docs,
        "chunks":            n_chunks,
        "chunk_size":        CHUNK_SIZE,
        "chunk_overlap":     CHUNK_OVERLAP,
        "embedding_backend": EMBEDDING_BACKEND if BUILD_VECTOR_INDEX else None,
        "embedding_modell":  EMBEDDING_MODEL if BUILD_VECTOR_INDEX else None,
        "collection_name":   COLLECTION_NAME,
        "vektor_index":      BUILD_VECTOR_INDEX,
    }

    with open(path, "w", encoding="utf-8") as f:
        json.dump(manifest, f, ensure_ascii=False, indent=2)

    print(f"📝 Manifest geschrieben: '{path}'")
    return manifest

## Pipeline ausführen

In [8]:
print("🚀 Starte Indexing-Pipeline...\n")

# --- Gemeinsamer Stamm ---
docs, n_docs = load_documents(DOC_SOURCE_DIR)
chunks       = split_documents(docs)

# --- Senke 1: semantisch ---
if BUILD_VECTOR_INDEX:
    print("\n--- Index 1: Vektordatenbank (ChromaDB) ---")
    db = create_vectorstore(chunks)

# --- Senke 2: lexikalisch ---
if BUILD_BM25_INDEX:
    print("\n--- Index 2: Lexikalischer Index (BM25) ---")
    #bm25_path = create_bm25_index(chunks)

# --- Manifest ---
print()
write_manifest(n_docs, len(chunks))

print("\n🎯 Nächster Schritt: Notebook 2 (RAG) öffnen und Fragen stellen.")

🚀 Starte Indexing-Pipeline...

  📄 20240604-Shortpaper_Standards-fuer-KI.pdf
  📄 2104.05314v2.pdf
  📄 EU-6-001-25-pdf.pdf
  📄 Guidelines_on_prohibited_artificial_intelligence_practices_established_by_Regulation_EU_20241689_AI_Act_German.PDF
  📄 Künstliche Intelligenz im Rahmen des AI Acts der EU.pdf
  📄 OJ_L_202401689_DE_TXT.pdf
  📄 Regulierung von KI _ Künstliche Intelligenz _ bpb.de.pdf
  📄 gutachten-datenethikkommission.pdf

✅ 649 Seiten/Abschnitte aus 8 Dokumenten geladen.
✅ 5953 Chunks erzeugt (Größe: 512, Overlap: 128)

--- Index 1: Vektordatenbank (ChromaDB) ---
🧠 Erstelle Embeddings (lmstudio) und speichere in ChromaDB...
✨ Vektor-Index gespeichert in './chroma_db'

📝 Manifest geschrieben: './index_manifest.json'

🎯 Nächster Schritt: Notebook 2 (RAG) öffnen und Fragen stellen.
